# 05 · One Segment as a QP

### Recap & why now
Every problem so far has had exactly as many conditions as coefficients, so the cost
matrix never had a job — the answer was forced. That changes here.

Give a segment more coefficients than conditions and there are infinitely many
trajectories that satisfy the specification. Choosing among them is an **optimisation**,
and the shape it takes — quadratic cost, linear constraints — is the shape every remaining
notebook in this project uses.

### Learning objectives
1. Recognise the standard **quadratic program** form and see that our problem is one.
2. Solve a quadratic cost with linear equality constraints through the **KKT system**.
3. Verify that the solver reproduces the closed-form septic when the problem is fully determined.
4. Watch the optimiser use its leftover freedom when the problem is not.
5. Explain why the problem is **convex**, and why that matters.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection used by the 3-D figures.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print numbers with 4 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=4, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Polynomial toolkit, built up over Notebooks 02-05 ===================

def poly_val(c, t, der=0):
    """Value of the polynomial c at time t, or of its `der`-th derivative."""
    out = 0.0
    for i in range(der, len(c)):                   # Terms below `der` differentiate away to zero.
        factor = 1.0
        for k in range(der):
            factor *= (i - k)                      # i(i-1)...(i-der+1), the falling factorial.
        out += c[i]*factor*t**(i - der)
    return out

def deriv_row(n, t, der):
    """Row r with r @ c = the der-th derivative at time t. One CONSTRAINT is one row."""
    r = np.zeros(n)
    for i in range(der, n):
        factor = 1.0
        for k in range(der):
            factor *= (i - k)
        r[i] = factor*t**(i - der)
    return r

def cost_matrix(n, T, der=4):
    """Q with c^T Q c = integral from 0 to T of (der-th derivative)^2 dt."""
    Q = np.zeros((n, n))
    for i in range(der, n):
        for j in range(der, n):
            ci = np.prod([i - k for k in range(der)])
            cj = np.prod([j - k for k in range(der)])
            power = i + j - 2*der + 1               # From integrating t^(i-der) * t^(j-der).
            Q[i, j] = ci*cj*T**power/power
    return Q

NCOEF = 8                                          # Order 7: eight coefficients, eight boundary conditions.
g = 9.81                                           # Gravity, needed whenever we turn acceleration into tilt.
print("polynomial toolkit ready — order %d, %d coefficients per segment per axis" % (NCOEF-1, NCOEF))

## 1 · The standard form

$$\min_c \;\; c^\top Q\, c \qquad \text{subject to} \qquad A c = b$$

A quadratic thing to minimise, linear things to obey. $Q$ is Notebook 04's snap cost;
$A c = b$ holds the boundary conditions from Notebook 02.

Two properties make this a comfortable place to be. $Q$ came from integrating a square,
so $c^\top Q c \ge 0$ always — the problem is **convex**, with no local minima to get
stuck in. And the constraints are flat, so the feasible set is a plane. A convex cost on
a plane has exactly one lowest point.

In [ ]:
n, T = 8, 1.0
Q = cost_matrix(n, T, der=4)                       # Snap cost, from Notebook 04.
print("Q is %d x %d, and its eigenvalues are all >= 0? min = %.2e ✔" %
      (*Q.shape, np.linalg.eigvalsh(Q).min()))

six = [(deriv_row(n, 0.0, d), 0.0) for d in range(3)] + \
      [(deriv_row(n, T, d), 1.0 if d == 0 else 0.0) for d in range(3)]
A = np.array([r for r, _ in six]); b = np.array([v for _, v in six])
print("\nA is %d x %d — six conditions, eight unknowns." % A.shape)
print("rank of A: %d, so the feasible set is a plane of dimension %d." %
      (np.linalg.matrix_rank(A), n - np.linalg.matrix_rank(A)))
print("\nEvery point on that plane is a valid trajectory. The cost picks one.")

## 2 · Solving it: Lagrange multipliers

At the optimum the cost cannot fall further **without breaking a constraint** — so its
gradient must be exactly balanced by the constraints pulling back. Writing that condition
alongside the constraints themselves gives one linear system:

$$\begin{bmatrix} 2Q & A^\top \\ A & 0 \end{bmatrix}
\begin{bmatrix} c \\ \lambda \end{bmatrix}
= \begin{bmatrix} 0 \\ b \end{bmatrix}$$

This is the **KKT system** (Karush–Kuhn–Tucker). $\lambda$ holds one multiplier per
constraint; we solve for it and throw it away.

One `np.linalg.solve` and the trajectory is done — no iteration, no tuning, no
convergence to worry about. An equality-constrained quadratic problem is just a bigger
linear system.

In [ ]:
def solve_qp(Q_, conditions):
    """Minimise c^T Q c subject to A c = b, through the KKT system."""
    A_ = np.array([r for r, _ in conditions]); b_ = np.array([v for _, v in conditions])
    n_ = Q_.shape[0]
    KKT = np.block([[2*Q_, A_.T], [A_, np.zeros((len(b_), len(b_)))]])
    sol = np.linalg.solve(KKT, np.concatenate([np.zeros(n_), b_]))
    return sol[:n_], sol[n_:]                      # Coefficients, then the multipliers we discard.

c_free, lam = solve_qp(Q, six)
print("six conditions, eight coefficients — the optimiser has two to spend.")
print("coefficients:", np.round(c_free, 4))
print("snap cost   : %.2f" % (c_free @ Q @ c_free))
print("\nverifying the conditions were actually met:")
for label, row, want in [("p(0)", deriv_row(n, 0, 0), 0.0), ("p(T)", deriv_row(n, T, 0), 1.0),
                         ("v(0)", deriv_row(n, 0, 1), 0.0), ("a(T)", deriv_row(n, T, 2), 0.0)]:
    print("   %s = %+.6f  (asked %.1f)" % (label, row @ c_free, want))

## 3 · Adding the last two conditions

Pin down the jerk at both ends as well and the count reaches eight — the problem becomes
fully determined, the leftover freedom disappears, and $Q$ becomes irrelevant.

The answer should then be exactly Notebook 04's septic, whatever cost you hand the
solver. That is a good test: an optimiser that changes its answer when the problem has
only one answer has a bug.

In [ ]:
eight = six + [(deriv_row(n, 0.0, 3), 0.0), (deriv_row(n, T, 3), 0.0)]
c_pinned, _ = solve_qp(Q, eight)
c_textbook = np.array([0, 0, 0, 0, 35.0, -84.0, 70.0, -20.0])
print("with eight conditions:", np.round(c_pinned, 4))
print("textbook septic      :", np.round(c_textbook, 4))
print("largest difference: %.2e ✔" % np.abs(c_pinned - c_textbook).max())

c_jerk_cost, _ = solve_qp(cost_matrix(n, T, der=3), eight)   # A DIFFERENT cost, same conditions.
print("\nsolved again with a JERK cost instead of a snap cost: %.2e difference" %
      np.abs(c_jerk_cost - c_pinned).max())
print("Identical, because there was nothing left to optimise. When the constraints determine")
print("the answer, the cost is decoration — which is worth knowing before you spend an afternoon")
print("tuning one.")

## 4 · What the freedom buys

Compare the two solutions. Both satisfy the same six conditions; the eight-condition one
also has zero jerk at the ends, and pays for it.

The free version wins on snap, exactly as Notebook 04's E1 argued — more constraints can
only raise a minimum. What the extra conditions buy is the clean handover at the
endpoints.

In [ ]:
print("  version                    snap cost   jerk at t=0   jerk at t=T")
for label, cc in [("six conditions (free) ", c_free), ("eight conditions      ", c_pinned)]:
    print("  %s %10.1f %13.2f %13.2f" %
          (label, cc @ Q @ cc, poly_val(cc, 0.0, 3), poly_val(cc, T, 3)))

tau = np.linspace(0, 1, 400)
fig, axes = plt.subplots(1, 4, figsize=(14, 2.7))
for d, ax_ in enumerate(axes):
    ax_.plot(tau, [poly_val(c_free, t_, d) for t_ in tau], color="C0", lw=2, label="6 conditions")
    ax_.plot(tau, [poly_val(c_pinned, t_, d) for t_ in tau], color="C3", lw=2, label="8 conditions")
    ax_.set_title(["position", "velocity", "acceleration", "jerk"][d], fontsize=10)
    ax_.set_xlabel(r"$\tau$")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("\nThe position curves are almost indistinguishable. The jerk plot is where they differ,")
print("and it differs exactly at the ends — which is the whole content of those two extra rows.")

## 🧪 Try it yourself

**E1.** The KKT system is square and symmetric but **not** positive definite — it has
negative eigenvalues. Why does `np.linalg.solve` still work?

**E2.** Solve with a cost that penalises acceleration instead of snap, using the same six
conditions, and compare the resulting curves.

In [ ]:
# --- Solution E1 ---
KKT = np.block([[2*Q, A.T], [A, np.zeros((6, 6))]])
ev = np.linalg.eigvalsh(KKT)
print("E1: the KKT matrix has %d negative and %d positive eigenvalues." %
      ((ev < 0).sum(), (ev > 0).sum()))
print("    np.linalg.solve does not require positive definiteness — it does LU with pivoting, which")
print("    works for any nonsingular matrix. What WOULD fail is Cholesky, which many QP solvers")
print("    reach for by default; they handle KKT systems with a symmetric-indefinite factorisation")
print("    instead. The negative eigenvalues are not a warning sign, they are structural: the")
print("    multipliers sit at a saddle point, not a minimum.")

# --- Solution E2 ---
c_accel, _ = solve_qp(cost_matrix(n, T, der=2), six)      # Penalise acceleration.
c_jerk_, _ = solve_qp(cost_matrix(n, T, der=3), six)      # ...or jerk.
print("\nE2:  cost minimised      peak accel   peak jerk   peak snap")
grid = np.linspace(0, T, 800)
for label, cc in [("acceleration    ", c_accel), ("jerk            ", c_jerk_), ("snap            ", c_free)]:
    peaks_ = [max(abs(poly_val(cc, t_, d)) for t_ in grid) for d in (2, 3, 4)]
    print("    %s %11.2f %11.2f %11.1f" % (label, *peaks_))
print("    Each wins on its own metric, as it must — that is what optimality means. The rows are")
print("    not a ranking; they are three answers to three different questions. Choosing which")
print("    question to ask is engineering, and Notebook 04 argued that for a quadcopter the")
print("    answer is snap.")

## 🚁 Mini-project: watching the freedom get spent

Animate the solution as the two extra conditions are gradually imposed — sweeping the
required end-jerk from a large value down to zero — and watch the curve reshape while the
position endpoints stay pinned.

In [ ]:
jerk_targets = np.concatenate([np.linspace(60, -60, 60), np.linspace(-60, 60, 60)])
curves = []
for j_end in jerk_targets:
    conds = six + [(deriv_row(n, 0.0, 3), 0.0), (deriv_row(n, T, 3), j_end)]
    cc, _ = solve_qp(Q, conds)
    curves.append(cc)

tau = np.linspace(0, 1, 200)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.0, 3.4))

def frame(k):
    a1.clear(); a2.clear()
    cc = curves[k]
    a1.plot(tau, [poly_val(cc, t_, 0) for t_ in tau], color="C0", lw=2.2)
    a1.plot([0, 1], [0, 1], "o", color="C3", ms=8)              # The pinned endpoints.
    a1.set_ylim(-0.3, 1.4); a1.set_xlabel(r"$\tau$"); a1.set_ylabel("position")
    a1.set_title("required jerk at the end = %+.0f" % jerk_targets[k], fontsize=10)
    a2.plot(tau, [poly_val(cc, t_, 3) for t_ in tau], color="C3", lw=2.2)
    a2.set_ylim(-200, 200); a2.set_xlabel(r"$\tau$"); a2.set_ylabel("jerk")
    a2.set_title("snap cost %.0f" % (cc @ Q @ cc), fontsize=10)
    return []

anim = animation.FuncAnimation(fig, frame, frames=len(curves), interval=45, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** Quadratic programs with linear constraints are everywhere in
> robotics, and always for the same reason: you can state what you want, and the solver
> either returns the unique best answer or tells you honestly that none exists. The same
> structure appears in model-predictive control, in whole-body control for legged robots,
> and in the thrust-allocation layer of rocket landing software. The modelling changes;
> the shape of the problem does not.

**Where next.** One segment is not a route. Notebook 06 chains several together, and the
interesting part is what happens where they meet.